In [0]:
%sh
mkdir -p /tmp/xena_raw
wget -O /tmp/xena_raw/expression.tsv.gz "https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-BRCA.star_fpkm-uq.tsv.gz"
wget -O /tmp/xena_raw/clinical.tsv.gz "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FBRCA_clinicalMatrix"
ls -lh /tmp/xena_raw/

In [0]:
%sh
echo "--- file types ---"
file /tmp/xena_raw/expression.tsv.gz
file /tmp/xena_raw/clinical.tsv.gz

echo "--- expression head (first 3 lines, first 5 fields) ---"
zcat /tmp/xena_raw/expression.tsv.gz 2>/dev/null | head -3 | cut -f1-5

echo "--- clinical: does PAM50Call_RNAseq exist? ---"
(zcat /tmp/xena_raw/clinical.tsv.gz 2>/dev/null || cat /tmp/xena_raw/clinical.tsv.gz) | head -1 | tr '\t' '\n' | grep -n -i pam50

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.xena_raw

In [0]:
%sh
# Volumes are FUSE-mounted, so a plain cp/mv works — no special API needed
cp /tmp/xena_raw/expression.tsv.gz /Volumes/workspace/default/xena_raw/expression.tsv.gz
cp /tmp/xena_raw/clinical.tsv.gz /Volumes/workspace/default/xena_raw/clinical.tsv   # dropped .gz — it's not actually compressed

ls -lh /Volumes/workspace/default/xena_raw/

In [0]:
expression_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("inferSchema", False) 
    .csv("/Volumes/workspace/default/xena_raw/expression.tsv.gz")
)

clinical_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .csv("/Volumes/workspace/default/xena_raw/clinical.tsv")
)

print(type(expression_df))

In [0]:
n_genes = expression_df.count()
n_samples = len(expression_df.columns) - 1  # minus the Ensembl_ID column
print(f"genes: {n_genes}, samples: {n_samples}")

In [0]:
from pyspark.sql import functions as F

sample_cols = [c for c in expression_df.columns if c != "Ensembl_ID"]

# 1. Cast the ~1,226 sample columns to double.
#    Still a transformation, not an action -- no execution yet.
expression_typed = expression_df.select(
    "Ensembl_ID",
    *[F.col(c).cast("double").alias(c) for c in sample_cols]
)

# Sanity check: are values already log-scale? Real FPKM-UQ maxes are usually
# in the thousands; log2(x+1) maxes are usually under ~20.
print(expression_typed.select(F.max(sample_cols[0])).collect())


In [0]:
# 2. MELT: wide (gene x 1,226 sample columns) -> long (Ensembl_ID, sample_barcode, expression_value)
#    stack() needs 'label, column' pairs listed explicitly -- with ~1,200 columns this
#    can trip Spark's codegen size limit (a "grows beyond 64 KB" WARNING). If you see that
#    warning, it's not fatal -- Spark just falls back to a slower execution path -- paste it
#    back and we'll switch to a chunked union approach if it's actually failing (not just warning).
n = len(sample_cols)
stack_cols = ", ".join([f"'{c}', `{c}`" for c in sample_cols])
melted = expression_typed.select(
    "Ensembl_ID",
    F.expr(f"stack({n}, {stack_cols}) as (sample_barcode, expression_value)")
)

In [0]:
# 3. Per-gene mean + variance in ONE groupBy pass over the long table.
#    This is the row-grouped aggregation Spark is built for.
gene_stats = melted.groupBy("Ensembl_ID").agg(
    F.mean("expression_value").alias("mean_expr"),
    F.variance("expression_value").alias("var_expr")
)

In [0]:
# 4. FILTER: drop low-expression genes first (adjust threshold once we see the
#    distribution -- 1.0 is a placeholder, not a tuned value)
LOW_EXPR_THRESHOLD = 1.0
TOP_N_GENES = 1500

filtered_stats = gene_stats.filter(F.col("mean_expr") >= LOW_EXPR_THRESHOLD)

# Rank by variance, keep top N gene IDs. .collect() here is a real action --
# fine, because the result is only ~1500 rows, not the full 60,660.
top_gene_ids = [
    row["Ensembl_ID"]
    for row in filtered_stats.orderBy(F.desc("var_expr")).limit(TOP_N_GENES).select("Ensembl_ID").collect()
]
print(f"Genes passing low-expression filter: {filtered_stats.count()}")
print(f"Top variance genes kept: {len(top_gene_ids)}")

In [0]:
# 5. Cut the long table down to just those genes, BEFORE pivoting (small data reshaped, not huge data)
melted_filtered = melted.filter(F.col("Ensembl_ID").isin(top_gene_ids))

# 6. PIVOT: long -> wide, transposed. Passing top_gene_ids explicitly avoids
#    Spark doing an extra distinct-values scan to figure out pivot columns itself.
patient_features = (
    melted_filtered
    .groupBy("sample_barcode")
    .pivot("Ensembl_ID", top_gene_ids)
    .agg(F.first("expression_value"))
)

print(patient_features.count(), len(patient_features.columns))

In [0]:
from pyspark.sql import functions as F

gene_cols = [c for c in patient_features.columns if c != "sample_barcode"]

null_counts = patient_features.select(
    [F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in gene_cols]
)

total_nulls = null_counts.select(
    sum([F.col(f"`{c}`") for c in gene_cols]).alias("total_nulls")
)
total_nulls.show()

dup_check = patient_features.groupBy("sample_barcode").count().filter(F.col("count") > 1)
print(f"Duplicate sample_barcode rows: {dup_check.count()}")

In [0]:
patient_features.select("sample_barcode", *[f"`{c}`" for c in gene_cols[:5]]).show(5, truncate=False)

In [0]:
import re

def strip_version(colname: str) -> str:
    return re.sub(r"\.\d+$", "", colname)

# Single select, one flat list of aliased columns -- not a loop of separate transformations
select_exprs = [
    F.col(f"`{c}`").alias(strip_version(c))
    for c in patient_features.columns
]
patient_features_clean = patient_features.select(*select_exprs)

n_cols_before = len(patient_features.columns)
n_cols_after = len(patient_features_clean.columns)
print(f"columns before: {n_cols_before}, after: {n_cols_after}")

patient_features_clean.select(
    "sample_barcode",
    *[c for c in patient_features_clean.columns if c != "sample_barcode"][:5]
).show(5)

In [0]:
sample_type_dist = (
    patient_features_clean
    .withColumn("sample_type_code", F.substring(
        F.element_at(F.split(F.col("sample_barcode"), "-"), -1), 1, 2
    ))
    .groupBy("sample_type_code")
    .count()
    .orderBy(F.desc("count"))
)
sample_type_dist.show()

In [0]:
from pyspark.sql import functions as F

gene_cols = [c for c in patient_features_clean.columns if c != "sample_barcode"]

patient_features_array = patient_features_clean.select(
    "sample_barcode",
    F.array(*[F.col(c) for c in gene_cols]).alias("gene_expression")
)

# keep the gene order as metadata so we know which array index maps to which gene later
gene_order_df = spark.createDataFrame(
    [(i, g) for i, g in enumerate(gene_cols)], ["array_index", "gene_id"]
)

print(patient_features_array.count(), len(patient_features_array.columns))
patient_features_array.show(3, truncate=100)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.brca_patient_features")

(
    patient_features_array
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.brca_patient_features")
)

spark.sql("DROP TABLE IF EXISTS workspace.default.brca_gene_order")
gene_order_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.brca_gene_order")

print("done")

In [0]:
%restart_python

In [0]:

from pyspark.ml.functions import array_to_vector
from pyspark.sql import functions as F
# Read the Delta table back -- this confirms round-tripping worked, and is
# also the first taste of why Delta > a plain file: we're reading from a
# governed table by name, not re-deriving anything or hunting for a file path
patient_features_delta = spark.table("workspace.default.brca_patient_features")

patient_features_vec = patient_features_delta.withColumn(
    "features", array_to_vector(F.col("gene_expression"))
)

patient_features_vec.printSchema()
patient_features_vec.select("sample_barcode", "features").show(3, truncate=80)

In [0]:
patient_features_delta = spark.table("workspace.default.brca_patient_features")

length_check = patient_features_delta.select(
    F.size("gene_expression").alias("arr_len")
).groupBy("arr_len").count().orderBy(F.desc("count"))

length_check.show()

In [0]:
from pyspark.ml.stat import Summarizer
from pyspark.sql import functions as F

summary_row = patient_features_vec.select(
    Summarizer.metrics("mean", "std").summary(F.col("features")).alias("summary")
).select("summary.mean", "summary.std").first()

mean_vec = summary_row["mean"]
std_vec = summary_row["std"]

print(f"mean_vec length: {len(mean_vec)}")
print(f"std_vec length: {len(std_vec)}")
print(f"first 5 means: {mean_vec[:5]}")
print(f"first 5 stds: {std_vec[:5]}")

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, DoubleType

zero_std_count = int((np.array(std_vec) == 0).sum())
print(f"genes with zero std: {zero_std_count}")

mean_np = np.array(mean_vec)
std_np = np.array(std_vec)

@pandas_udf(ArrayType(DoubleType()))
def scale_array(arr_series: pd.Series) -> pd.Series:
    # mean_np / std_np captured directly via closure -- fine at ~12KB each
    return arr_series.apply(lambda arr: ((np.array(arr) - mean_np) / std_np).tolist())

patient_features_scaled = patient_features_vec.withColumn(
    "scaled_array", scale_array(F.col("gene_expression"))
).withColumn(
    "scaled_features", array_to_vector(F.col("scaled_array"))
)

patient_features_scaled.select("sample_barcode", "scaled_features").show(3, truncate=80)

In [0]:
patient_features_scaled.select("sample_barcode", "scaled_features").write.format("delta").mode("overwrite").saveAsTable("workspace.default.brca_patient_scaled")

print(spark.table("workspace.default.brca_patient_scaled").count())

In [0]:
dbutils.widgets.text("n_components", "10", "PCA components")
dbutils.widgets.text("k_clusters", "4", "KMeans k")
dbutils.widgets.text("top_n_genes", "1500", "Top N genes (for MLflow logging only -- not re-run here)")

N_COMPONENTS = int(dbutils.widgets.get("n_components"))
K_CLUSTERS = int(dbutils.widgets.get("k_clusters"))
TOP_N_GENES = int(dbutils.widgets.get("top_n_genes"))

print(f"N_COMPONENTS={N_COMPONENTS}, K_CLUSTERS={K_CLUSTERS}, TOP_N_GENES={TOP_N_GENES}")

In [0]:
from pyspark.ml.functions import vector_to_array
import numpy as np
import pandas as pd

scaled_delta = spark.table("workspace.default.brca_patient_scaled")

pdf = scaled_delta.select(
    "sample_barcode",
    vector_to_array("scaled_features").alias("scaled_array")
).toPandas()

barcodes = pdf["sample_barcode"].values
X = np.stack(pdf["scaled_array"].values)  # shape: (n_patients, n_genes)

print(f"X shape: {X.shape}")
print(f"X dtype: {X.dtype}")
print(f"X mean (should be ~0): {X.mean():.4f}")
print(f"X std (should be ~1): {X.std():.4f}")

In [0]:
# economy SVD: since n_patients (1226) < n_genes (1500), U is 1226x1226,
# S has 1226 singular values, Vt is 1226x1500
U, S, Vt = np.linalg.svd(X, full_matrices=False)

# top N_COMPONENTS directions (rows of Vt = principal component axes in gene space)
V_top = Vt[:N_COMPONENTS, :].T          # shape: (1500, N_COMPONENTS)
pca_scores = X @ V_top                   # shape: (1226, N_COMPONENTS) -- same as pca_features would've been

# explained variance ratio, same quantity pca_model.explainedVariance gave you
explained_var = (S ** 2) / np.sum(S ** 2)
explained_var_top = explained_var[:N_COMPONENTS]

print(f"pca_scores shape: {pca_scores.shape}")
print(f"explained variance (top {N_COMPONENTS}): {np.round(explained_var_top, 4)}")
print(f"cumulative explained variance: {explained_var_top.sum():.4f}")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

# rebuild a Spark DataFrame from the driver-side PCA result
pca_pdf = pd.DataFrame(pca_scores, columns=[f"pc{i+1}" for i in range(N_COMPONENTS)])
pca_pdf.insert(0, "sample_barcode", barcodes)

patient_pca_spark = spark.createDataFrame(pca_pdf)

pc_cols = [f"pc{i+1}" for i in range(N_COMPONENTS)]
assembler = VectorAssembler(inputCols=pc_cols, outputCol="pca_features")
patient_pca_vec = assembler.transform(patient_pca_spark)

kmeans = KMeans(featuresCol="pca_features", predictionCol="cluster", k=K_CLUSTERS, seed=42)
kmeans_model = kmeans.fit(patient_pca_vec)
patient_clustered = kmeans_model.transform(patient_pca_vec)

patient_clustered.groupBy("cluster").count().orderBy("cluster").show()

In [0]:
import mlflow

with mlflow.start_run(run_name="brca_pca_kmeans_v1") as run:
    mlflow.log_param("top_n_genes", TOP_N_GENES)
    mlflow.log_param("low_expr_threshold", 1.0)
    mlflow.log_param("n_components", N_COMPONENTS)
    mlflow.log_param("k_clusters", K_CLUSTERS)
    mlflow.log_param("pca_method", "manual_svd_numpy")
    mlflow.log_param("scaler_method", "manual_summarizer_pandas_udf")

    mlflow.log_metric("cumulative_explained_variance", float(explained_var_top.sum()))
    for i, v in enumerate(explained_var_top):
        mlflow.log_metric(f"explained_variance_pc{i+1}", float(v))

    mlflow.spark.log_model(
        kmeans_model,
        artifact_path="kmeans_model",
        dfs_tmpdir="/Volumes/workspace/default/xena_raw/mlflow_tmp"
    )

    run_id = run.info.run_id
    print(f"MLflow run_id: {run_id}")

patient_clustered.select("sample_barcode", *pc_cols, "cluster").write.format("delta").mode("overwrite").saveAsTable("workspace.default.brca_patient_clustered")

print(spark.table("workspace.default.brca_patient_clustered").count())

In [0]:
clinical_raw = spark.read.option("header", "true").option("sep", "\t").csv(
    "/Volumes/workspace/default/xena_raw/clinical.tsv"
)

print(f"row count: {clinical_raw.count()}")
print(f"column count: {len(clinical_raw.columns)}")

# confirm the PAM50 column exists and check its value distribution
clinical_raw.groupBy("PAM50Call_RNAseq").count().orderBy(F.desc("count")).show(20, truncate=False)

In [0]:
clinical_raw.select("sampleID").show(5, truncate=False)

# compare barcode lengths between the two sources
print("clinical sampleID length distribution:")
clinical_raw.select(F.length("sampleID").alias("len")).groupBy("len").count().show()

print("expression sample_barcode length distribution:")
spark.table("workspace.default.brca_patient_clustered") \
    .select(F.length("sample_barcode").alias("len")).groupBy("len").count().show()

In [0]:
patient_clustered = spark.table("workspace.default.brca_patient_clustered")

patient_clustered_key = patient_clustered.withColumn(
    "join_key", F.substring("sample_barcode", 1, 15)
)

clinical_pam50 = clinical_raw.select(
    F.col("sampleID").alias("join_key"),
    "PAM50Call_RNAseq"
)

patient_with_pam50 = patient_clustered_key.join(
    clinical_pam50, on="join_key", how="left"
).drop("join_key")

print(f"total rows after join: {patient_with_pam50.count()}")
patient_with_pam50.groupBy("PAM50Call_RNAseq").count().orderBy(F.desc("count")).show()

# cross-tab: does sample type (from barcode suffix) explain the nulls, as expected?
patient_with_pam50.withColumn(
    "sample_type", F.substring("sample_barcode", 14, 2)
).groupBy("sample_type", "PAM50Call_RNAseq").count().orderBy("sample_type", F.desc("count")).show(20)

In [0]:
patient_with_pam50_final = patient_clustered_key.join(
    clinical_pam50, on="join_key", how="left"
).drop("join_key").withColumn(
    "sample_type", F.substring("sample_barcode", 14, 2)
)

patient_with_pam50_final.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.brca_patient_pam50"
)

print(spark.table("workspace.default.brca_patient_pam50").count())
spark.table("workspace.default.brca_patient_pam50").select(
    "sample_barcode", "cluster", "PAM50Call_RNAseq", "sample_type"
).show(5, truncate=False)

In [0]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="pca_features", predictionCol="cluster", metricName="silhouette"
)

# ClusteringEvaluator needs the vector column -- rebuild it the same way Stage 4 did
patient_clustered_full = spark.table("workspace.default.brca_patient_pam50")
pc_cols_check = [f"pc{i+1}" for i in range(N_COMPONENTS)]

assembler_eval = VectorAssembler(inputCols=pc_cols_check, outputCol="pca_features")
patient_for_eval = assembler_eval.transform(patient_clustered_full)

silhouette = evaluator.evaluate(patient_for_eval)
print(f"silhouette score: {silhouette:.4f}")


In [0]:
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("silhouette_score", float(silhouette))

print("logged silhouette to run:", run_id)

# contingency table: cluster vs PAM50, tumor samples only (sample_type = '01')
# -- excluding normal/metastatic here since PAM50 "Normal-like" isn't a tumor subtype call
contingency = patient_clustered_full.filter(
    (F.col("sample_type") == "01") & F.col("PAM50Call_RNAseq").isNotNull()
).groupBy("cluster", "PAM50Call_RNAseq").count()

contingency_pivot = contingency.groupBy("cluster").pivot("PAM50Call_RNAseq").sum("count").fillna(0).orderBy("cluster")
contingency_pivot.show()

In [0]:
from sklearn.metrics import adjusted_rand_score

tumor_only = patient_clustered_full.filter(
    (F.col("sample_type") == "01") & F.col("PAM50Call_RNAseq").isNotNull()
).select("cluster", "PAM50Call_RNAseq").toPandas()

ari = adjusted_rand_score(tumor_only["PAM50Call_RNAseq"], tumor_only["cluster"])
print(f"n samples used: {len(tumor_only)}")
print(f"Adjusted Rand Index: {ari:.4f}")

with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("adjusted_rand_index", float(ari))
    

In [0]:
# pull raw expression + cluster assignment together, joined on sample_barcode
raw_with_cluster = spark.table("workspace.default.brca_patient_features").join(
    spark.table("workspace.default.brca_patient_clustered").select("sample_barcode", "cluster"),
    on="sample_barcode"
)

raw_pdf = raw_with_cluster.select("sample_barcode", "cluster", "gene_expression").toPandas()

X_raw = np.stack(raw_pdf["gene_expression"].values)   # shape: (1226, 1500)
cluster_labels = raw_pdf["cluster"].values

print(f"X_raw shape: {X_raw.shape}")
print(f"cluster label counts: {np.unique(cluster_labels, return_counts=True)}")

In [0]:
from scipy import stats as scipy_stats

k = K_CLUSTERS
N = X_raw.shape[0]

grand_mean = X_raw.mean(axis=0)                       # shape: (1500,) -- overall mean per gene

group_means = np.zeros((k, X_raw.shape[1]))
ss_between = np.zeros(X_raw.shape[1])
ss_within = np.zeros(X_raw.shape[1])

for c in range(k):
    mask = cluster_labels == c
    n_c = mask.sum()
    group_mean_c = X_raw[mask].mean(axis=0)
    group_means[c] = group_mean_c
    ss_between += n_c * (group_mean_c - grand_mean) ** 2
    ss_within += ((X_raw[mask] - group_mean_c) ** 2).sum(axis=0)

df_between = k - 1
df_within = N - k
f_stat = (ss_between / df_between) / (ss_within / df_within)

# sanity check: compare our manual F-stat against scipy's for one gene
gene0_groups = [X_raw[cluster_labels == c, 0] for c in range(k)]
scipy_f, scipy_p = scipy_stats.f_oneway(*gene0_groups)
print(f"gene 0 -- manual F: {f_stat[0]:.4f}, scipy F: {scipy_f:.4f}  (should match)")

print(f"F-stat range across all genes: min={f_stat.min():.2f}, max={f_stat.max():.2f}")
print(f"top 10 F-stats: {np.sort(f_stat)[::-1][:10].round(1)}")

In [0]:
p_values = scipy_stats.f.sf(f_stat, df_between, df_within)  # survival function = 1 - CDF, more numerically stable than 1-cdf for small p

biomarker_pdf = pd.DataFrame({
    "array_index": np.arange(X_raw.shape[1]),
    "f_stat": f_stat,
    "p_value": p_values,
})

for c in range(k):
    biomarker_pdf[f"mean_cluster{c}"] = group_means[c]

biomarker_pdf["mean_overall"] = grand_mean

# map array_index back to Ensembl gene IDs
gene_order = spark.table("workspace.default.brca_gene_order").toPandas()
biomarker_full = biomarker_pdf.merge(gene_order, on="array_index", how="left")

biomarker_full = biomarker_full.sort_values("f_stat", ascending=False).reset_index(drop=True)

print(f"biomarker table shape: {biomarker_full.shape}")
print(f"any unmapped array_index (null gene_id)? {biomarker_full['gene_id'].isnull().sum()}")
biomarker_full.head(15)[["gene_id", "f_stat", "p_value", "mean_cluster0", "mean_cluster1", "mean_cluster2", "mean_cluster3", "mean_overall"]]

In [0]:
biomarker_spark = spark.createDataFrame(biomarker_full)
biomarker_spark.write.format("delta").mode("overwrite").saveAsTable("workspace.default.brca_biomarkers")

with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("top_biomarker_f_stat", float(biomarker_full["f_stat"].iloc[0]))
    mlflow.log_param("top_biomarker_gene_id", biomarker_full["gene_id"].iloc[0])

print(spark.table("workspace.default.brca_biomarkers").count())